# AI Data Engineer — Feature Store & DQ

> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)

Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.
Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.

## Hogyan futtasd

```bash
# 1. Virtuális környezet (Python 3.10+)
python -m venv .venv

# Windows:
.venv\Scripts\activate

# macOS/Linux:
source .venv/bin/activate

# 2. Telepítsd a függőségeket (a notebook első cellája)

# 3. Indítsd a Jupytert
jupyter lab
# vagy
jupyter notebook
```

Minden cella saját magában értelmezhető. A `# %%` kommentek Jupyterben és VS Code-ban is a cellák határát jelölik.


## 1. Környezet


In [ ]:
%pip install pandas pyarrow "great-expectations>=1.0" --quiet

## 2. Feature engineering — Web

Shop churnKészítsünk néhány feature-t az ügyfelek lemorzsolódásának előrejelzésére.


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Szimulált forrás: rendelések + ügyfelek
now = datetime(2025, 5, 1)
orders = pd.DataFrame({
    'order_id': range(1, 11),
    'customer_id': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4],
    'amount': [12500, 8900, 3400, 24000, 11000, 5200, 7800, 15000, 950, 1200],
    'created_at': [now - timedelta(days=d) for d in [200, 150, 5, 300, 40, 180, 90, 10, 400, 380]],
})

# Feature-k ügyfélenként
features = (
    orders.groupby('customer_id').agg(
        total_orders=('order_id', 'count'),
        total_revenue=('amount', 'sum'),
        avg_order_value=('amount', 'mean'),
        first_order_at=('created_at', 'min'),
        last_order_at=('created_at', 'max'),
    )
)
features['days_since_last_order'] = (now - features['last_order_at']).dt.days
features['customer_lifetime_days'] = (now - features['first_order_at']).dt.days
features['churn_risk'] = (features['days_since_last_order'] > 90).astype(int)
features.reset_index()


## 3. Offline / online feature store szimuláció

**Offline store** (Parquet) — ML traininghez, nagy volumenű historikus adatokhoz.

**Online store** (Redis / DynamoDB / RocksDB) — ML servinghez, milliszekundumos válaszidővel.


In [ ]:
# Offline → Parquet
features.to_parquet('features_offline.parquet')
print('Offline store: features_offline.parquet')

# Online → in-memory dict (valós életben Redis)
online = features.to_dict(orient='index')

def get_features_for_serving(customer_id: int) -> dict:
    """ML serving time lookup — ~ms latency."""
    return online.get(customer_id, {})

# Mintha egy API hívás jönne be
print('Customer 1 feature-ei serving time-ban:')
f = get_features_for_serving(1)
for k, v in list(f.items())[:5]:
    print(f'  {k}: {v}')


## 4. Great Expectations — adatminőség szabályok


In [ ]:
import great_expectations as gx

context = gx.get_context(mode='ephemeral')
ds = context.data_sources.add_pandas('orders_ds')
asset = ds.add_dataframe_asset(name='orders_asset')
batch = asset.add_batch_definition_whole_dataframe('all_orders').get_batch({'dataframe': orders})

suite = context.suites.add(gx.ExpectationSuite(name='orders_quality'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='order_id'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column='order_id'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column='amount', min_value=0, max_value=1_000_000))

result = batch.validate(suite)
print('Összes expectation:', result.statistics['evaluated_expectations'])
print('Sikeres:', result.statistics['successful_expectations'])
print('Sikertelen:', result.statistics['unsuccessful_expectations'])


## 5. Train/serve skew elkerülése

A leggyakoribb ML bug: tanításkor egy feature értékét `X` módon számolod, servingkor `Y` módon, ezért ugyanarra a customerre más jön ki.

**Megoldás:** egyetlen **feature transzformációs függvény** legyen, amit tanításkor és servingkor is ugyanúgy használsz.


In [ ]:
def compute_customer_features(orders: pd.DataFrame, customer_id: int, now: datetime) -> dict:
    """Egyetlen forrás — tanításkor és servingkor is ezt hívjuk."""
    o = orders[orders['customer_id'] == customer_id]
    if len(o) == 0:
        return {}
    return {
        'total_orders': len(o),
        'total_revenue': float(o['amount'].sum()),
        'avg_order_value': float(o['amount'].mean()),
        'days_since_last_order': (now - o['created_at'].max()).days,
    }

# Training time — batch
training = [compute_customer_features(orders, cid, now) for cid in orders['customer_id'].unique()]
print('Training batch:')
for t in training[:2]:
    print(' ', t)

# Serving time — single record
print('\nServing (customer 1):', compute_customer_features(orders, 1, now))


## Következő lépések

- Térj vissza a [web-alapú kurzushoz](./index.html) a teljes anyagért, diagramokért és kvízekért.
- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.
- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)

---

*Engineering Crash Courses · MIT License · Magyar Data & AI Engineering kurzusok*
